# Section 7: Macrophage Metagene Analysis

## Purpose
Identifies two macrophage populations — **High-Metagene (TAMs)** and **Low-Metagene (SAMs)** —
using a 19-gene signature derived from Wilcoxon DE of KSHV⁺ macrophages in tumor-associated
vs stromal niches (bootstrap-validated, 5 × 5 000-cell subsamples).

## Workflow Overview

| Step | Description |
|------|-------------|
| 1 | Load data |
| 2 | Compute metagene score |
| 3 | Threshold (Otsu → user-adjustable) |
| 4 | PHATE visualization |
| 5 | Bootstrap DE validation + export |


## 1. Imports

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import scipy.sparse as sp
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

import scanpy as sc
import scanpy.external as scext
from skimage.filters import threshold_otsu

sc.settings.verbosity = 1


In [ ]:
from tqdm import tqdm
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.cm as cm
import os
import pandas as pd
import numpy as np
from sklearn.cluster import DBSCAN
import alphashape
from scipy.interpolate import splprep, splev
from shapely.geometry import Polygon
from geopandas import GeoDataFrame

def plot_spatial_feature_grid_with_spline(
    adata,
    path_block_cores,
    features,
    feature_color_palettes=None,
    spline_feature=None,
    spline_categories=None,
    selected_features_dict=None,
    cmap=cm.viridis,
    linewidth=0.5,
    edgecolor='k',
    spline_alpha=0.07,
    dbscan_eps=50,
    dbscan_min_samples=5,
    spline_smoothing=2.0,
    spline_line_color='blue',
    spline_linewidth=3,
    background_cell_color='#d3d3d3',
    show=True,
    save_path="../figures/grid_spline/"
):

    def convert2gpd(df):
        grouped = df.groupby('cell_id')
        polygons = []
        for cell_id, group in grouped:
            points = group[['vertex_x', 'vertex_y']].values
            if len(points) > 2:
                poly = Polygon(points)
                polygons.append({'cell_id': cell_id, 'geometry': poly})
        gdf = GeoDataFrame(polygons)
        return gdf

    n_rows, n_cols = len(features), len(path_block_cores)
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 5 * n_rows), dpi=300)

    if n_rows == 1 and n_cols == 1:
        axes = [[axes]]
    elif n_rows == 1:
        axes = [axes]
    elif n_cols == 1:
        axes = [[ax] for ax in axes]

    for i, feature in enumerate(tqdm(features, desc="Features")):
        for j, core_id in enumerate(tqdm(path_block_cores, desc="  Plotting cores", leave=False)):
            ax = axes[i][j]

            core_adata = adata[adata.obs['path_block_core'] == core_id]
            if core_adata.n_obs == 0:
                print(f"[WARNING] No cells for {core_id}")
                ax.axis('off')
                continue

            sample = core_adata.obs['sample_id'].unique()[0]
            stage = core_adata.obs['Stage'].unique()[0]
            boundary_path = f"../data/cell_boundaries/{sample}/cell_boundaries.csv.gz"
            boundaries_df = pd.read_csv(boundary_path, compression='gzip')
            core_df = boundaries_df[boundaries_df['cell_id'].isin(core_adata.obs['cell_id'])]
            gdf = convert2gpd(core_df)

            columns_to_merge = ['cell_id']
            for col in [feature, spline_feature]:
                if col and col not in columns_to_merge:
                    columns_to_merge.append(col)
            gdf = gdf.merge(core_adata.obs[columns_to_merge], on='cell_id', how='left')

            is_categorical = True
            if feature_color_palettes and feature in feature_color_palettes:
                palette = feature_color_palettes[feature]
                selected = selected_features_dict.get(feature) if selected_features_dict else None
                gdf[feature] = gdf[feature].astype(str)
                if selected:
                    gdf['color'] = gdf[feature].apply(
                        lambda val: palette[val] if val in selected and val in palette else background_cell_color
                    )
                else:
                    gdf['color'] = gdf[feature].map(palette).fillna(background_cell_color)
            else:
                is_categorical = False
                vmin, vmax = gdf[feature].min(), gdf[feature].max()
                if vmin == vmax:
                    vmin, vmax = vmin - 0.1, vmax + 0.1
                norm = mcolors.Normalize(vmin=vmin, vmax=vmax)
                gdf['color'] = gdf[feature].apply(
                    lambda x: mcolors.to_hex(cmap(norm(x))) if pd.notnull(x) else background_cell_color
                )

            gdf.plot(color=gdf['color'], ax=ax, linewidth=linewidth, edgecolor=edgecolor)
            ax.axis('off')
            ax.invert_yaxis()

            # -------- Spline overlay with debug logs --------
            if spline_feature:
                if spline_feature not in gdf.columns:
                    print(f"[ERROR] spline_feature '{spline_feature}' not found in gdf for {core_id}")
                    continue
                gdf[spline_feature] = gdf[spline_feature].astype(str)
                print(f"[INFO] Unique values in '{spline_feature}' for {core_id}: {gdf[spline_feature].unique()}")

                cats = spline_categories or gdf[spline_feature].unique().tolist()
                for cat in cats:
                    cat_gdf = gdf[gdf[spline_feature] == cat]
                    print(f"[DEBUG] Spline category '{cat}': {len(cat_gdf)} cells in {core_id}")
                    if cat_gdf.empty:
                        print(f"[WARNING] No cells found for spline category '{cat}' in {core_id}")
                        continue

                    centroids = np.vstack([geom.centroid.coords[0] for geom in cat_gdf.geometry])
                    labels = DBSCAN(eps=dbscan_eps, min_samples=dbscan_min_samples).fit_predict(centroids)
                    cat_gdf = cat_gdf.assign(cluster=labels)

                    # print(f"[DEBUG] Clusters found for '{cat}': {set(labels)}")
                    # ax.scatter(centroids[:, 0], centroids[:, 1], s=5, color='blue', alpha=0.5)

                    for cl in set(labels):
                        if cl < 0: continue
                        sub = cat_gdf[cat_gdf.cluster == cl]
                        all_pts = []
                        for geom in sub.geometry:
                            pieces = geom.geoms if geom.geom_type == 'MultiPolygon' else [geom]
                            for poly in pieces:
                                all_pts.extend(poly.exterior.coords)
                        if not all_pts:
                            print(f"[WARNING] No boundary points for cluster {cl} in {cat}")
                            continue
                        hull = alphashape.alphashape(all_pts, spline_alpha)
                        if hull.is_empty:
                            print(f"[WARNING] Alpha shape empty for {cat} cluster {cl}")
                            continue
                        polys = [hull] if hull.geom_type == 'Polygon' else list(hull.geoms)
                        for poly in polys:
                            x, y = poly.exterior.xy
                            # ax.plot(x, y, color='lime', linewidth=2)  # Debug outline
                            pts = np.vstack((x, y)).T
                            if len(pts) >= 3:
                                try:
                                    tck, _ = splprep([pts[:, 0], pts[:, 1]], s=spline_smoothing, per=True)
                                    out = splev(np.linspace(0, 1, 200), tck)
                                    ax.plot(out[0], out[1], linewidth=spline_linewidth, color=spline_line_color)
                                except Exception as e:
                                    print(f"[ERROR] Spline fitting failed: {e}")
                                    ax.plot(x, y, linewidth=spline_linewidth, color=spline_line_color)

    plt.tight_layout()
    os.makedirs(save_path, exist_ok=True)
    feature_str = "_".join(features)
    core_count = len(path_block_cores)
    out_path = os.path.join(save_path, f"spatial_grid_spline_{feature_str}_n{core_count}.png")

    plt.savefig(out_path, dpi=300)
    if show:
        plt.show()
    else:
        plt.close()
    print(f"[INFO] Saved to {out_path}")


## 2. Load data

In [ ]:
# adata = sc.read_h5ad('../data/KS_adata_preprocessed.h5ad')

adata = sc.read_h5ad('../data/KS_adata_preprocessed.h5ad')
adata.obs['niches'] = adata.obs['niche_with_tumor_proximity'].copy()
adata.obsm['spatial'] = adata.obs[['local_x', 'local_y']].to_numpy()
print(f'Full dataset: {adata.n_obs:,} cells × {adata.n_vars} genes')

## 3. Metagene score

19-gene signature: top-ranked genes from Wilcoxon DE of KSHV⁺ macrophages,
tumor-associated niches vs stromal/immune/skin niches (KSHV.K2 excluded — 
down-regulated in tumor niches, mean logFC = −0.53 across 5 bootstrap runs).


In [ ]:
METAGENE_GENES = [
    "KSHV.ORF71", "KSHV.ORF72", "PROX1",   "ECSCR",  "GNG11",
    "CDH5",       "CD34",       "KDR",      "RAMP2",  "FSCN1",
    "CALCRL",     "LYVE1",      "MYCT1",    "SPRY1",  "TFPI",
    "PDGFA",      "VWF",        "HSPG2",    "KIT",
]

available = [g for g in METAGENE_GENES if g in adata.var_names]
missing   = [g for g in METAGENE_GENES if g not in adata.var_names]
print(f'Available: {len(available)}/{len(METAGENE_GENES)} genes')
if missing:
    print(f'Missing  : {missing}')


In [ ]:
# Compute mean log-normalised expression across metagene genes for every macrophage
mac_mask = adata.obs['broad_cell_types'] == 'Macrophages'
Xm = adata[mac_mask, available].X
if sp.issparse(Xm):
    Xm = Xm.toarray()
metagene_score = Xm.mean(axis=1)

adata.obs['metagene_score'] = np.nan
adata.obs.loc[mac_mask, 'metagene_score'] = metagene_score

print(f'Macrophages scored: {mac_mask.sum():,}')
print(f'Score  min={metagene_score.min():.4f}  '
      f'median={np.median(metagene_score):.4f}  '
      f'max={metagene_score.max():.4f}')


## 4. Threshold determination

Otsu's method on the score histogram provides a data-driven starting point.
Adjust `THRESHOLD` below after inspecting the plot.


In [ ]:
otsu_threshold = threshold_otsu(metagene_score)

otsu_threshold = 0.06
print(f'Otsu threshold: {otsu_threshold:.4f}')

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.hist(metagene_score, bins=150, color='#4c72b0', alpha=0.75, linewidth=0)
ax.axvline(otsu_threshold, color='firebrick', linewidth=1.5,
           linestyle='--', label=f'Otsu = {otsu_threshold:.4f}')
ax.set_xlabel('Mean metagene expression (log-normalised)', fontsize=10)
ax.set_ylabel('Cells', fontsize=10)
ax.set_title('Macrophage metagene score distribution', fontsize=11)
ax.legend(fontsize=9)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()


In [ ]:
# ── USER: adjust threshold here ───────────────────────────────────────────────
THRESHOLD = otsu_threshold   # override, e.g. 
THRESHOLD = 0.06
# ──────────────────────────────────────────────────────────────────────────────

# Initialize as object dtype (None) — np.nan creates float64 which rejects strings
adata.obs['metagene_cluster'] = None
adata.obs.loc[mac_mask, 'metagene_cluster'] = np.where(
    metagene_score >= THRESHOLD, 'High-Metagene', 'Low-Metagene'
)

counts = adata.obs['metagene_cluster'].value_counts()
print(f'Threshold used : {THRESHOLD:.4f}')
print(counts.to_string())
print(f'High / total   : {counts.get("High-Metagene", 0) / counts.sum():.1%}')


## 5. PHATE visualization

Transfer metagene annotations from the full dataset onto the pre-computed
macrophage + LEC PHATE embedding (matched by cell index / cell_id).


In [ ]:
# Transfer metagene annotations: full adata → PHATE subset (join on cell index)
score_lookup   = adata.obs['metagene_score'].dropna()
cluster_lookup = adata.obs['metagene_cluster'].dropna()

adata_phate.obs['metagene_score']   = adata_phate.obs.index.map(score_lookup)
adata_phate.obs['metagene_cluster'] = adata_phate.obs.index.map(cluster_lookup)

phate_mac_mask = adata_phate.obs['broad_cell_types'] == 'Macrophages'
n_annotated = adata_phate.obs.loc[phate_mac_mask, 'metagene_cluster'].notna().sum()
print(f'PHATE macrophages annotated: {n_annotated:,} / {phate_mac_mask.sum():,}')


In [ ]:
palette = {'High-Metagene': '#ff40ff', 'Low-Metagene': '#f5d600'}

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, feat, title, cmap in zip(
    axes,
    ['metagene_cluster', 'metagene_score'],
    ['Metagene cluster', 'Metagene score'],
    [None, 'viridis'],
):
    sub = adata_phate.obs.copy()
    coords = adata_phate.obsm['X_phate']

    if feat == 'metagene_cluster':
        for label, color in [('Low-Metagene', '#f5d600'), ('High-Metagene', '#ff40ff')]:
            sel = sub[feat] == label
            ax.scatter(coords[sel, 0], coords[sel, 1],
                       c=color, s=0.4, linewidths=0, alpha=0.6,
                       rasterized=True, label=label)
        na_sel = sub[feat].isna()
        ax.scatter(coords[na_sel, 0], coords[na_sel, 1],
                   c='lightgray', s=0.2, linewidths=0, alpha=0.3,
                   rasterized=True, label='LECs / unannotated')
        ax.legend(markerscale=8, fontsize=8, framealpha=0.7)
    else:
        vals = sub[feat].values.astype(float)
        vmin, vmax = np.nanpercentile(vals, [2, 98])
        sc_plot = ax.scatter(coords[:, 0], coords[:, 1],
                             c=vals, cmap=cmap, vmin=vmin, vmax=vmax,
                             s=0.4, linewidths=0, alpha=0.6,
                             rasterized=True)
        plt.colorbar(sc_plot, ax=ax, fraction=0.046, pad=0.04).set_label(
            'Mean expression', fontsize=8)

    ax.set_title(title, fontsize=11)
    ax.set_xlabel('PHATE 1', fontsize=9)
    ax.set_ylabel('PHATE 2', fontsize=9)
    ax.set_xticks([]); ax.set_yticks([])
    ax.spines[['top','right','left','bottom']].set_visible(False)

fig.suptitle('Macrophage metagene state on PHATE embedding\n'
             '(macrophage + LEC subset; KSHV clusters excluded)',
             fontsize=12)
plt.tight_layout()
plt.savefig('../figures/macrophage_metagene_PHATE.pdf', dpi=300, bbox_inches='tight')
plt.savefig('../figures/macrophage_metagene_PHATE.png', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
ctypes_with_m1m2_colors = {
    'Lymphatic Endothelial Cells': '#d3d3d3',
    'Vascular Endothelial Cells': '#d3d3d3',
    'Pericytes': '#d3d3d3',
    'Fibroblasts': '#d3d3d3',
    'T-cells': '#d3d3d3',
    'Keratinocytes': '#d3d3d3',
    'Dendritic cells': '#d3d3d3',
    'Spinous to Granular Cells': '#d3d3d3',
    'Pilosebaceous Cells': '#d3d3d3',
    'B-cells': '#d3d3d3',
    'Melanocytes': '#d3d3d3',
    'Low-Metagene': 'green',
    'High-Metagene': '#ff40ff',
    #'Low-Metagene': 'yellow',
    #'High-Metagene': '#ff40ff',
    
}
path_block_cores = ['SYS23-969_A_4', 'SYD11-8594_2A_1', 'SYD21-8311_A_1', 'SYD19-2015_A_5']



plot_spatial_feature_grid_with_spline(
    adata=adata,
    # path_block_cores=['SYD06-2641_2D_3', 'SYD19-2015_A_5'],
    path_block_cores=path_block_cores,
    features=['metagene_cluster'],
    feature_color_palettes={'metagene_cluster': ctypes_with_m1m2_colors},
    selected_features_dict={'metagene_cluster': ['High-Metagene', 'Low-Metagene']},
    spline_feature='niches_grouped',
    spline_categories=['tumor_associated'],
    spline_line_color='blue',
    spline_linewidth=1, 
    dbscan_eps=19, 
    dbscan_min_samples=5,
    linewidth=0.05,
    background_cell_color='white',
    save_path='../figures/figure_7_1/',
    show=True
)


## 6. Bootstrap DE validation

Wilcoxon rank-sum (scanpy), KSHV⁺ macrophages only, tumor-associated vs rest.
5 runs × 5 000 cells per niches_grouped category (all 79 skin cells kept).
p-values: geometric mean across runs (avoids inflation from full-N DE).


In [ ]:
N_SUBSAMPLE = 10000
N_BOOTS     = 10
SEED        = 42

mac_inf = adata[
    (adata.obs['broad_cell_types'] == 'Macrophages') &
    (adata.obs['infection_status'] == 'infected')
].copy()
print(f'KSHV+ macrophages: {mac_inf.n_obs:,}')
print(mac_inf.obs['niches_grouped'].value_counts().to_string())


In [ ]:
rng  = np.random.default_rng(SEED)
runs = []

for b in range(N_BOOTS):
    idx = []
    for grp, sub in mac_inf.obs.groupby('niches_grouped', observed=True):
        n = min(N_SUBSAMPLE, len(sub))
        idx.extend(rng.choice(sub.index, size=n, replace=False).tolist())

    sub_ad = mac_inf[idx].copy()
    sc.tl.rank_genes_groups(sub_ad, groupby='niches_grouped',
                            method='wilcoxon', tie_correct=True)
    df = sc.get.rank_genes_groups_df(sub_ad, group=['tumor_associated'])
    runs.append(df.set_index('names'))
    print(f'  Boot {b+1}: {sub_ad.n_obs} cells | '
          + ' | '.join(f'{g}={v}' for g,v in
                       sub_ad.obs['niches_grouped'].value_counts().items()),
          flush=True)


In [ ]:
eps = np.finfo(float).tiny

lfc   = pd.concat([r['logfoldchanges'].rename(f'lfc_{i}')  for i,r in enumerate(runs)], axis=1)
score = pd.concat([r['scores'].rename(f'score_{i}')        for i,r in enumerate(runs)], axis=1)
pval  = pd.concat([r['pvals'].rename(f'pval_{i}')          for i,r in enumerate(runs)], axis=1)
padj  = pd.concat([r['pvals_adj'].rename(f'padj_{i}')      for i,r in enumerate(runs)], axis=1)

result = pd.DataFrame(index=runs[0].index)
result.index.name = 'gene'
result['mean_logFC']        = lfc.mean(axis=1)
result['std_logFC']         = lfc.std(axis=1)
result['mean_score']        = score.mean(axis=1)
result['geomean_pval']      = np.exp(np.log(pval.clip(lower=eps)).mean(axis=1))
result['geomean_pvals_adj'] = np.exp(np.log(padj.clip(lower=eps)).mean(axis=1))

result_sorted = (result.sort_values('mean_logFC', ascending=False)
                       .reset_index())
result_sorted.insert(0, 'rank_by_logFC', range(1, len(result_sorted)+1))

print('Top 25 by mean logFC:')
print(result_sorted[['gene','mean_logFC','std_logFC',
                      'geomean_pval','geomean_pvals_adj']].head(25).to_string(index=False))


In [ ]:
# Metagene-gene rows with their original list rank
lfc_rank   = result_sorted.set_index('gene')['rank_by_logFC']
top_result = result.reindex(METAGENE_GENES).reset_index()
top_result.insert(0, 'rank', range(1, len(METAGENE_GENES)+1))
top_result['rank_by_logFC'] = top_result['gene'].map(lfc_rank)

import os
os.makedirs('supplementary_tables', exist_ok=True)
out = '../supplementary_tables/Table_Sx_macrophage_metagene_DE_bootstrap.csv'
top_result.to_csv(out, index=False)
print(f'Saved → {out}')
print(top_result.to_string(index=False))


## 7. Full DE table (all 308 genes)

In [ ]:
out_full = '../supplementary_tables/Table_Sx_macrophage_metagene_DE_bootstrap_full.csv'
result_sorted.to_csv(out_full, index=False)
print(f'Saved → {out_full}')
result_sorted.head(20)


## 8. Bootstrap DE — All macrophages (KSHV⁺ and KSHV⁻), tumor-associated vs rest

In [ ]:
mac_all = adata[adata.obs['broad_cell_types'] == 'Macrophages'].copy()
print(f'All macrophages: {mac_all.n_obs:,}')
print(mac_all.obs['niches_grouped'].value_counts().to_string())


In [ ]:
rng_all  = np.random.default_rng(SEED)
runs_all = []

for b in range(N_BOOTS):
    idx = []
    for grp, sub in mac_all.obs.groupby('niches_grouped', observed=True):
        n = min(N_SUBSAMPLE, len(sub))
        idx.extend(rng_all.choice(sub.index, size=n, replace=False).tolist())

    sub_ad = mac_all[idx].copy()
    sc.tl.rank_genes_groups(sub_ad, groupby='niches_grouped',
                            method='wilcoxon', tie_correct=True)
    df = sc.get.rank_genes_groups_df(sub_ad, group=['tumor_associated'])
    runs_all.append(df.set_index('names'))
    print(f'  Boot {b+1}: {sub_ad.n_obs} cells | '
          + ' | '.join(f'{g}={v}' for g, v in
                       sub_ad.obs['niches_grouped'].value_counts().items()),
          flush=True)


In [ ]:
lfc_a   = pd.concat([r['logfoldchanges'].rename(f'lfc_{i}')  for i, r in enumerate(runs_all)], axis=1)
score_a = pd.concat([r['scores'].rename(f'score_{i}')        for i, r in enumerate(runs_all)], axis=1)
pval_a  = pd.concat([r['pvals'].rename(f'pval_{i}')          for i, r in enumerate(runs_all)], axis=1)
padj_a  = pd.concat([r['pvals_adj'].rename(f'padj_{i}')      for i, r in enumerate(runs_all)], axis=1)

result_all = pd.DataFrame(index=runs_all[0].index)
result_all.index.name = 'gene'
result_all['mean_logFC']        = lfc_a.mean(axis=1)
result_all['std_logFC']         = lfc_a.std(axis=1)
result_all['mean_score']        = score_a.mean(axis=1)
result_all['geomean_pval']      = np.exp(np.log(pval_a.clip(lower=eps)).mean(axis=1))
result_all['geomean_pvals_adj'] = np.exp(np.log(padj_a.clip(lower=eps)).mean(axis=1))

result_all_sorted = (result_all.sort_values('mean_logFC', ascending=False)
                               .reset_index())
result_all_sorted.insert(0, 'rank_by_logFC', range(1, len(result_all_sorted)+1))

print('Top 25 by mean logFC (all macrophages):')
print(result_all_sorted[['gene','mean_logFC','std_logFC',
                          'geomean_pval','geomean_pvals_adj']].head(25).to_string(index=False))

out_all = '../supplementary_tables/Table_Sx_allMac_DE_bootstrap.csv'
result_all_sorted.to_csv(out_all, index=False)
print(f'\nSaved → {out_all}')


## 9. Bootstrap DE — KSHV⁺ macrophages in tumor vs KSHV⁻ macrophages outside tumor

Compares two biologically distinct populations:
- **inf_tumor**: KSHV⁺ macrophages (`infection_status == 'infected'`) in `tumor_associated` niches
- **uninf_nontumor**: KSHV⁻ macrophages (`infection_status == 'uninfected'`) in non-tumor niches

Bootstrap: 10 runs × 10 000 cells per group.

In [ ]:
mac_obs = mac_all.obs.copy()

inf_tumor_mask      = (mac_obs['infection_status'] == 'infected')   & (mac_obs['niches_grouped'] == 'tumor_associated')
uninf_nontumor_mask = (mac_obs['infection_status'] == 'uninfected') & (mac_obs['niches_grouped'] != 'tumor_associated')

mac_obs['de_group'] = None
mac_obs.loc[inf_tumor_mask,      'de_group'] = 'inf_tumor'
mac_obs.loc[uninf_nontumor_mask, 'de_group'] = 'uninf_nontumor'

print('Group sizes:')
print(mac_obs['de_group'].value_counts(dropna=True).to_string())

# Subset to the two groups only
keep = mac_obs['de_group'].notna()
mac_cross = mac_all[keep].copy()
mac_cross.obs['de_group'] = mac_obs.loc[keep, 'de_group'].values
print(f'\nSubset for cross-infection DE: {mac_cross.n_obs:,} cells')


In [ ]:
rng_cross  = np.random.default_rng(SEED)
runs_cross = []

for b in range(N_BOOTS):
    idx = []
    for grp, sub in mac_cross.obs.groupby('de_group', observed=True):
        n = min(N_SUBSAMPLE, len(sub))
        idx.extend(rng_cross.choice(sub.index, size=n, replace=False).tolist())

    sub_ad = mac_cross[idx].copy()
    sc.tl.rank_genes_groups(sub_ad, groupby='de_group',
                            groups=['inf_tumor'],
                            reference='uninf_nontumor',
                            method='wilcoxon', tie_correct=True)
    df = sc.get.rank_genes_groups_df(sub_ad, group=['inf_tumor'])
    runs_cross.append(df.set_index('names'))
    print(f'  Boot {b+1}: {sub_ad.n_obs} cells | '
          + ' | '.join(f'{g}={v}' for g, v in
                       sub_ad.obs['de_group'].value_counts().items()),
          flush=True)


In [ ]:
lfc_c   = pd.concat([r['logfoldchanges'].rename(f'lfc_{i}')  for i, r in enumerate(runs_cross)], axis=1)
score_c = pd.concat([r['scores'].rename(f'score_{i}')        for i, r in enumerate(runs_cross)], axis=1)
pval_c  = pd.concat([r['pvals'].rename(f'pval_{i}')          for i, r in enumerate(runs_cross)], axis=1)
padj_c  = pd.concat([r['pvals_adj'].rename(f'padj_{i}')      for i, r in enumerate(runs_cross)], axis=1)

result_cross = pd.DataFrame(index=runs_cross[0].index)
result_cross.index.name = 'gene'
result_cross['mean_logFC']        = lfc_c.mean(axis=1)
result_cross['std_logFC']         = lfc_c.std(axis=1)
result_cross['mean_score']        = score_c.mean(axis=1)
result_cross['geomean_pval']      = np.exp(np.log(pval_c.clip(lower=eps)).mean(axis=1))
result_cross['geomean_pvals_adj'] = np.exp(np.log(padj_c.clip(lower=eps)).mean(axis=1))

result_cross_sorted = (result_cross.sort_values('mean_logFC', ascending=False)
                                   .reset_index())
result_cross_sorted.insert(0, 'rank_by_logFC', range(1, len(result_cross_sorted)+1))

print('Top 25 by mean logFC (KSHV+ tumor vs KSHV- outside tumor):')
print(result_cross_sorted[['gene','mean_logFC','std_logFC',
                            'geomean_pval','geomean_pvals_adj']].head(25).to_string(index=False))

out_cross = '../supplementary_tables/Table_Sx_crossInfection_DE_bootstrap.csv'
result_cross_sorted.to_csv(out_cross, index=False)
print(f'\nSaved → {out_cross}')
